## Preparing the dataset for modelling (NOT READY YET)

### general paths and libraries

In [ ]:
!pip install seaborn

In [ ]:
import os
import numpy as np
import rasterio
from rasterio import mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
# variables - Best results (pearson correlation with LST) for window size
lst_clipped_path = '../../datos-notebooks/0.LST/LST_clipped_celsius.tif'
ndwi_1_path = '../../datos-notebooks/1.land-cover/moving-windows/ndwi/Class_1_window61.tif'
ndvi_1_path = '../../datos-notebooks/1.land-cover/moving-windows/ndvi/Class_1_window25.tif'
veg_vol_path = '../../datos-notebooks/2.objects-height/moving-windows/tree-volume/trees_window13.tif'
built_vol_path = '../../datos-notebooks/2.objects-height/moving-windows/building-volume/buildings_window13.tif'
built_area_path = '../../datos-notebooks/2.objects-height/moving-windows/building-area/building_area_window11.tif'
building_height_path = '../../datos-notebooks/2.objects-height/moving-windows/building-height/building_height_window21.tif'
mean_elev_path = '../../datos-notebooks/3.topography/moving_windows/elev_mean/elev_mean_win111.tif'
mean_slope_path = '../../datos-notebooks/3.topography/moving_windows/slope_mean/slope_mean_win19.tif'
northness_path = '../../datos-notebooks/3.topography/moving_windows/northness/northness_win201.tif'

output_dir = '../../datos-notebooks/4.LST-prediction/'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
with rasterio.open(lst_clipped_path) as src:
    lst = src.read(1)

lst_mask = np.isnan(lst)

# mask values out of city boundary
ndwi_1 = np.where(lst_mask, np.nan, rasterio.open(ndwi_1_path).read(1).astype("float32"))
ndvi_1 = np.where(lst_mask, np.nan, rasterio.open(ndvi_1_path).read(1).astype("float32"))
veg_vol = np.where(lst_mask, np.nan, rasterio.open(veg_vol_path).read(1).astype("float32"))
built_vol = np.where(lst_mask, np.nan, rasterio.open(built_vol_path).read(1).astype("float32"))
built_area = np.where(lst_mask, np.nan, rasterio.open(built_area_path).read(1).astype("float32"))
built_height = np.where(lst_mask, np.nan, rasterio.open(building_height_path).read(1).astype("float32"))
mean_elev = np.where(lst_mask, np.nan, rasterio.open(mean_elev_path).read(1).astype("float32"))
mean_slope = np.where(lst_mask, np.nan, rasterio.open(mean_slope_path).read(1).astype("float32"))
northness = np.where(lst_mask, np.nan, rasterio.open(northness_path).read(1).astype("float32"))

rasters_masked = {
    "Water": ndwi_1,
    "Dense vegetation presence": ndvi_1,
    "Vegetation volume": veg_vol,
    "Built volume": built_vol,
    "Built area": built_area,
    "Building height": built_height,
    "Elevation": mean_elev,
    "Slope": mean_slope,
    "Northness": northness
}


In [ ]:
#plot masked rasters 

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, (name, data) in zip(axes.flat, rasters_masked.items()):

    img = ax.imshow(data, cmap='viridis')
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

In [ ]:
# Plot histograms of masked rasters
fig, axes = plt.subplots(3, 3, figsize=(16, 8))
axes = axes.flatten()  # Flatten to 1D for easy indexing

for i, (name, arr) in enumerate(rasters_masked.items()):
    # Plot histogram using already-masked arrays
    axes[i].hist(arr[~np.isnan(arr)], bins=50)
    axes[i].set_title(name)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")

# Hide unused subplots (if less than 8 rasters)
for j in range(i + 1, 8):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


### standarize variables - mean:0, std:1

In [ ]:
output_dir = '../../datos-notebooks/4.LST-prediction/standarized_variables'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Z-score standardization for regression predictors

def standardize_array(arr, profile, out_path):
    mean = np.nanmean(arr)
    std = np.nanstd(arr)
    z = (arr - mean) / std

    profile.update(dtype='float32')

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(z.astype('float32'), 1)

In [ ]:
# Apply standarization
with rasterio.open(lst_clipped_path) as src:
    profile = src.profile.copy()
    
standardize_array(ndwi_1, profile, os.path.join(output_dir, 'NDWI_1_z.tif'))
standardize_array(ndvi_1, profile, os.path.join(output_dir, 'NDVI_1_z.tif'))
standardize_array(veg_vol, profile, os.path.join(output_dir, 'VEG_VOL_z.tif'))
standardize_array(built_vol, profile, os.path.join(output_dir, 'BUILT_VOL_z.tif'))
standardize_array(built_area, profile, os.path.join(output_dir, 'BUILT_AREA_z.tif'))
standardize_array(built_height, profile, os.path.join(output_dir, 'BUILDING_HEIGHT_z.tif'))
standardize_array(mean_elev, profile, os.path.join(output_dir, 'ELEV_z.tif'))
standardize_array(mean_slope, profile, os.path.join(output_dir, 'SLOPE_z.tif'))
standardize_array(northness, profile, os.path.join(output_dir, 'NORTH_z.tif'))

In [ ]:
std_rasters_paths = {
    "Water": os.path.join(output_dir, "NDWI_1_z.tif"),
    "Dense veg. presence": os.path.join(output_dir, "NDVI_1_z.tif"),
    "Vegetation Volume": os.path.join(output_dir, "VEG_VOL_z.tif"),
    "Built Volume": os.path.join(output_dir, "BUILT_VOL_z.tif"),
    "Built Area": os.path.join(output_dir, "BUILT_AREA_z.tif"),
    "Built Height": os.path.join(output_dir, "BUILDING_HEIGHT_z.tif"),
    "Elevation": os.path.join(output_dir, "ELEV_z.tif"),
    "Slope": os.path.join(output_dir, "SLOPE_z.tif"),
    "Northness": os.path.join(output_dir, "NORTH_z.tif")
}

In [ ]:

#plot standardized rasters

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, (name, path) in zip(axes.flat, std_rasters_paths.items()):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)

    img = ax.imshow(data, cmap='viridis')
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(3, 3, figsize=(16, 8))
axes = axes.flatten()  # Flatten to 1D for easy indexing

for i, (name, path) in enumerate(std_rasters_paths.items()):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata

    # Plot histogram
    axes[i].hist(arr[~np.isnan(arr)], bins=50)
    axes[i].set_title(name)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")
    axes[i].axvline(0, linestyle='--')

plt.tight_layout()
plt.show()

### stratified sampling of LST

In [ ]:
rasters = {"LST": lst_clipped_path} # create dictionary with LST raster (not standardized)
rasters.update(std_rasters_paths) # add standardized rasters to the dictionary
rasters.pop("Elevation", None)
rasters.pop("Built Volume", None)


In [ ]:
n_bins = 10

In [ ]:
# Read and flatten
data_arrays = {}
for name, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1)
        data_arrays[name] = arr.flatten()

df = pd.DataFrame(data_arrays)

# Drop rows with NaN
df = df.dropna()

# Create LST bins for stratification (5 bins)

df['LST_bin'] = pd.qcut(df['LST'], n_bins, labels=False)


In [ ]:
df['LST_bin'].value_counts().sort_index()


In [ ]:
# Stratified sampling: 5% of each bin
sample_fraction = 0.1
df_sampled = df.groupby('LST_bin', group_keys=False).apply(
    lambda x: x.sample(frac=sample_fraction, random_state=42)
)

# Drop bin column
df_sampled = df_sampled.drop(columns=['LST_bin']).reset_index(drop=True)

print(df_sampled.head())
print("Sampled dataset shape:", df_sampled.shape)


# Split features and target
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']



In [ ]:
# Recreate bins for the sampled dataset
df_sampled['LST_bin'] = pd.qcut(df_sampled['LST'], n_bins, labels=False)

# Show min and max LST in each bin
for b in range(n_bins):
    values_in_bin = df_sampled[df_sampled['LST_bin'] == b]['LST']
    print(f"Bin {b}: min = {values_in_bin.min():.2f}, max = {values_in_bin.max():.2f}, count = {len(values_in_bin)}")



In [ ]:
# Compute correlation matrix (only features)
corr_matrix = X.corr()

# Display heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# --- Vegetation plot ---
plt.figure()
plt.scatter(
    df_sampled["Dense veg. presence"],
    df_sampled["Vegetation Volume"],
    s=1, alpha=0.5
)
plt.xlabel("Vegetation presence (z)")
plt.ylabel("Vegetation volume (z)")
plt.title("Vegetation: Area vs Volume")
plt.show()

# --- Built plot ---
plt.figure()
plt.scatter(
    df_sampled["Built Area"],
    df_sampled["Built Height"],
    s=1, alpha=0.5
)
plt.xlabel("Built area (z)")
plt.ylabel("Built Height (z)")
plt.title("Built: Area vs Height")
plt.show()


### Descriptive statistics

In [ ]:
lst_vals = df_sampled["LST"]
predictors = [v for v in df_sampled.columns if v != "LST"]
n_vars = len(predictors)


fig, axes = plt.subplots(3, 3, figsize=(10, 12))
axes = axes.flatten()  # para indexar más fácil

for i, v in enumerate(predictors):
    axes[i].scatter(df_sampled[v], lst_vals, s=1, alpha=0.3)
    axes[i].set_xlabel(v)
    axes[i].set_ylabel("LST")
    axes[i].set_title(f"LST vs {v}")

plt.tight_layout()
plt.show()

### PCA

In [ ]:
import rasterio
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ---------------------------
# Load rasters
# ---------------------------
data_arrays = []
predictor_names = []  # names excluding LST
profile_ref = None

for name, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if name == "LST":
            lst = arr.copy()        # keep LST separate
            profile_ref = src.profile
        else:
            data_arrays.append(arr)
            predictor_names.append(name)

# Stack predictors to (n_pixels, n_variables)
stack = np.stack(data_arrays, axis=-1)
nrows, ncols, nbands = stack.shape
X = stack.reshape(-1, nbands)

# ---------------------------
# Mask invalid pixels (use LST as reference)
# ---------------------------
valid_mask = ~np.isnan(lst.reshape(-1))
X_valid = X[valid_mask]

# ---------------------------
# Standardize variables
# ---------------------------
scaler = StandardScaler()
X_std = scaler.fit_transform(X_valid)

# ---------------------------
# PCA
# ---------------------------
pca = PCA()
X_pca = pca.fit_transform(X_std)

# ---------------------------
# Explained variance
# ---------------------------
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative variance:", np.cumsum(pca.explained_variance_ratio_))

# ---------------------------
# Save first 4 PCs as rasters
# ---------------------------
for i in range(4):
    pc_full = np.full(X.shape[0], np.nan)
    pc_full[valid_mask] = X_pca[:, i]

    pc_img = pc_full.reshape(nrows, ncols)

    out_path = os.path.join(output_dir, f"PC{i+1}.tif")
    profile = profile_ref.copy()
    profile.update(dtype=rasterio.float32, count=1, compress="lzw")

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(pc_img.astype(np.float32), 1)

    print(f"Saved: {out_path}")

# ---------------------------
# Loadings (variable weights per PC)
# ---------------------------
print("\nLoadings (PC weights):")
for i, comp in enumerate(pca.components_):
    print(f"\nPC{i+1}")
    for name, weight in zip(predictor_names, comp):
        print(f"{name:12s}: {weight:.3f}")



In [ ]:
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.ylim(0,1)
plt.title("Cumulative variance")
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, v in enumerate(df.columns):
    axes[i].boxplot(df[v], vert=False)
    axes[i].set_title(v)

for ax in axes[len(df.columns):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# LST vector for valid pixels
lst_vals = lst.reshape(-1)[valid_mask]

# Use first 8 PCs
pcs_to_plot = X_pca[:, :7]
pc_labels = [f"PC{i+1}" for i in range(8)]

# Compare PC1 against PC2..PC8
pc1_index = 0
other_pcs = range(1, 8)

# Create subplots (7 plots)
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for ax, j in zip(axes, other_pcs):
    sc = ax.scatter(pcs_to_plot[:, pc1_index], pcs_to_plot[:, j],
                    c=lst_vals, s=1, cmap="inferno", alpha=0.5)
    ax.set_xlabel(pc_labels[pc1_index])
    ax.set_ylabel(pc_labels[j])
    ax.set_title(f"{pc_labels[pc1_index]} vs {pc_labels[j]}")

# Hide extra subplots
for ax in axes[len(other_pcs):]:
    ax.axis("off")

plt.suptitle("PC1 vs Other PCs (colored by LST)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create DataFrame with first 6 PCs
loadings = pd.DataFrame(pca.components_[:6],
                        columns=predictor_names,
                        index=[f"PC{i+1}" for i in range(6)])

# Bar plot
loadings.T.plot(kind="bar", figsize=(12, 6))
plt.ylabel("Loading value")
plt.title("PCA Loadings (PC1–PC6)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### stepwise based on AIC

In [ ]:
!pip install statsmodels
import numpy as np
import statsmodels.api as sm

In [ ]:
# Target variable (LST)
y = lst_vals  # already masked with valid_mask

# Predictors: first 8 PCs
X = X_pca[:, :8]

# Add constant (intercept)
X_full = sm.add_constant(X)

# Names
pc_names = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7"]

# Forward stepwise selection using AIC (more strict)
selected = []
remaining = pc_names.copy()
current_aic = np.inf

# Settings
min_delta = 1000    # Minimum AIC improvement required
max_vars = 7   # Maximum number of PCs allowed

while remaining and len(selected) < max_vars:
    aic_with_candidates = []
    
    for candidate in remaining:
        vars_idx = [pc_names.index(v) for v in selected + [candidate]]
        X_test = X_full[:, [0] + [i+1 for i in vars_idx]]  # intercept + PCs
        model = sm.OLS(y, X_test).fit()
        aic_with_candidates.append((model.aic, candidate))
    
    aic_with_candidates.sort()
    best_aic, best_candidate = aic_with_candidates[0]
    
    # Only accept if AIC improves enough
    if current_aic - best_aic > min_delta:
        current_aic = best_aic
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        print(f"Added: {best_candidate}, AIC = {best_aic:.2f}")
    else:
        print("No candidate improved AIC enough. Stopping.")
        break

print("\nSelected PCs:", selected)



In [ ]:
# construction of matrix for model fitting
final_idx = [pc_names.index(v) for v in selected] 
X_final = X_full[:, [0] + [i+1 for i in final_idx]]


# fits an Ordinary Least Squares (OLS) regression model 
final_model = sm.OLS(y, X_final).fit()

print(final_model.summary())

In [ ]:
import matplotlib.pyplot as plt
import math

# LST values for valid pixels
lst_vals = lst.reshape(-1)[valid_mask]

# PCs seleccionadas
pcs_selected = X_final[:, 1:]  # omitimos la constante
pc_names_selected = selected
n_pcs = pcs_selected.shape[1]

# Definir layout de 2 filas
n_cols = math.ceil(n_pcs / 2)
fig, axes = plt.subplots(2, n_cols, figsize=(4*n_cols, 8))
axes = axes.flatten()

for i, pc_name in enumerate(pc_names_selected):
    ax = axes[i]
    sc = ax.scatter(pcs_selected[:, i], lst_vals, s=1, alpha=0.5, c=lst_vals, cmap='inferno')
    ax.set_xlabel(pc_name)
    ax.set_ylabel("LST")
    ax.set_title(f"{pc_name} vs LST")

# Quitar ejes vacíos si hay
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
cbar = fig.colorbar(sc, ax=axes.tolist(), fraction=0.02)
cbar.set_label("LST")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Coeficientes de las PCs (omitimos el intercepto)
coefs = final_model.params[1:]  # x1, x2, ..., x8
pc_names_selected = selected    # nombres de PCs seleccionadas

# Importancia relativa (valor absoluto)
importance = np.abs(coefs)

# Plot
plt.figure(figsize=(8,5))
plt.bar(pc_names_selected, importance, color='skyblue')
plt.ylabel("Importance (|coef|)")
plt.title("Relative contribution of selected PCs to LST")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# not updated
'''
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# Crear DataFrame de loadings
loadings = {
    "PC1": [0.028, 0.452, 0.262, -0.335, -0.348, 0.403, 0.420, 0.392],
    "PC2": [-0.150, 0.089, 0.408, 0.536, 0.510, 0.340, 0.336, -0.164],
    "PC3": [0.936, -0.029, 0.207, 0.132, 0.127, -0.076, -0.058, 0.195],
    "PC4": [-0.033, 0.091, 0.728, -0.214, -0.225, -0.336, -0.136, -0.482],
    "PC5": [-0.309, 0.054, 0.337, 0.091, 0.210, -0.285, -0.389, 0.709],
    "PC6": [0.050, 0.878, -0.270, 0.174, 0.114, -0.243, -0.164, -0.160],
    "PC7": [-0.025, -0.051, -0.072, -0.158, 0.232, -0.648, 0.695, 0.100],
    "PC8": [-0.037, -0.056, 0.004, 0.688, -0.664, -0.209, 0.164, 0.101]
}

variables = ["Water", "Dense veg.", "Veg Vol", "Built Vol", "Built Area", "Elev", "Slope", "Northness"]

df_loadings = pd.DataFrame(loadings, index=variables)

plt.figure(figsize=(10,6))
sns.heatmap(df_loadings, annot=True, cmap="coolwarm", center=0)
plt.title("PC Loadings")
plt.show()
'''